# 02.3 — Spatial features, hotel infrastructure and EDA

This notebook starts from the rebuilt `resorts_clean.csv` (target: 280 resorts, 70 per country). It geocodes resorts with OpenStreetMap Nominatim, calculates great-circle distances to six major cities, counts mapped 4- and 5-star hotels from real Overpass results, and produces price/size EDA. Missing coordinates or unavailable Overpass results remain missing; no Alpine-centroid or price/slope proxy is used.

In [ ]:
import os
import re
import time
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

DATA_DIR = "../data/processed"
INPUT_CSV = os.path.join(DATA_DIR, "resorts_clean.csv")
FEATURES_CSV = os.path.join(DATA_DIR, "resorts_features.csv")
GEOCODE_CACHE_CSV = os.path.join(DATA_DIR, "resorts_geocoded_cache.csv")
HOTEL_CACHE_CSV = os.path.join(DATA_DIR, "hotels_cache.csv")
os.makedirs(DATA_DIR, exist_ok=True)

df = pd.read_csv(INPUT_CSV)
if len(df) != 280 or not df.groupby("country").size().eq(70).all():
    raise ValueError("Run 02.2 first: expected 280 resorts with 70 per country.")

HEADERS = {"User-Agent": "ZHAW-AlpineSkiResearch/1.0 (academic project; contact via repository owner)"}
CITIES = {
    "zurich": (47.3769, 8.5417),
    "geneva": (46.2044, 6.1432),
    "milan": (45.4642, 9.1900),
    "turin": (45.0703, 7.6869),
    "munich": (48.1351, 11.5820),
    "lyon": (45.7640, 4.8357),
}


def normalise_name(value):
    value = str(value).replace("\u200b", "").replace("\u200c", "").replace("\u200d", "")
    value = value.replace("\u2060", "").replace("\ufeff", "").replace("’", "'")
    value = unicodedata.normalize("NFKD", value)
    value = "".join(ch for ch in value if not unicodedata.combining(ch))
    value = re.sub(r"[^a-zA-Z0-9]+", " ", value).lower()
    return re.sub(r"\s+", " ", value).strip()


def haversine_km(lat1, lon1, lat2, lon2):
    radius_km = 6371.0088
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat, dlon = lat2 - lat1, lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * radius_km * np.arcsin(np.sqrt(a))

## 1. Geocoding with a persistent, name-normalised cache

The cache is matched using a normalised resort name (invisible Unicode characters, accents and punctuation are removed). New locations are requested from Nominatim with a one-second pause. There is no default coordinate: unresolved resorts keep `NaN` and are excluded from distance and hotel calculations.

In [ ]:
cache_columns = [
    "resort_name", "country", "latitude", "longitude", "geocode_query",
    "geocode_status", "geocode_display_name", "geocode_osm_type",
    "geocode_osm_id", "geocode_class", "geocode_type",
]
if os.path.exists(GEOCODE_CACHE_CSV):
    geo_cache = pd.read_csv(GEOCODE_CACHE_CSV)
else:
    geo_cache = pd.DataFrame(columns=cache_columns)

# The previous cache contains several coordinates that are clearly inconsistent
# with their resort locations. Refresh those records instead of silently reusing them.
REGEOCODE_QUERY_OVERRIDES = {
    normalise_name("4 Vallees – Verbier/La Tzoumaz/Nendaz/Veysonnaz/Thyon"): "Verbier, Valais, Switzerland",
    normalise_name("Diedamskopf – Schoppernau"): "Schoppernau, Vorarlberg, Austria",
    normalise_name("Espace Lumiere – Pra Loup/Val d'Allos"): "Pra Loup, Uvernet-Fours, France",
    normalise_name("Espace San Bernardo – La Rosiere/La Thuile"): "La Rosiere, Montvalezan, France",
    normalise_name("Grossglockner Resort Kals-Matrei"): "Kals am Grossglockner, Tyrol, Austria",
    normalise_name("Ischgl/Samnaun – Silvretta Arena"): "Ischgl, Tyrol, Austria",
    normalise_name("Le Grand Domaine – Valmorel/Doucy/Celliers/St Francois Longchamp"): "Valmorel, Les Avanchers-Valmorel, France",
    normalise_name("Le Lys – Cauterets"): "Cauterets, Hautes-Pyrenees, France",
    normalise_name("Les Angles"): "Les Angles, Pyrenees-Orientales, France",
    normalise_name("Les Arcs/Peisey-Vallandry (Paradiski)"): "Les Arcs, Bourg-Saint-Maurice, France",
    normalise_name("Les Brasses"): "Les Brasses, Saint-Jeoire, Haute-Savoie, France",
    normalise_name("Les Portes du Mont-Blanc – Combloux/Megeve le Jaillet/La Giettaz"): "Combloux, Haute-Savoie, France",
    normalise_name("Piau-Engaly"): "Piau-Engaly, Aragnouet, Hautes-Pyrenees, France",
    normalise_name("Puy-Saint-Vincent"): "Puy-Saint-Vincent, Hautes-Alpes, France",
    normalise_name("Ramsau am Dachstein – Rittisberg"): "Ramsau am Dachstein, Styria, Austria",
    normalise_name("Rauriser Hochalmbahnen – Rauris"): "Rauris, Salzburg, Austria",
    normalise_name("Roc d'Enfer – St. Jean d'Aulps-La Grande Terche/Bellevaux-La Chevrerie"): "Saint-Jean-d'Aulps, Haute-Savoie, France",
    normalise_name("Saint-Lary-Soulan"): "Saint-Lary-Soulan, Hautes-Pyrenees, France",
    normalise_name("Sainte-Foy-Tarentaise"): "Sainte-Foy-Tarentaise, Savoie, France",
    normalise_name("Steinplatte Winklmoosalm – Waidring/Reit im Winkl"): "Waidring, Tyrol, Austria",
    normalise_name("Val Senales Glacier (Schnalstaler Gletscher)"): "Senales, South Tyrol, Italy",
}
regeocode_keys = set(REGEOCODE_QUERY_OVERRIDES)

geo_cache["name_key"] = geo_cache["resort_name"].map(normalise_name)
cache_lookup = {}
for _, cached in geo_cache.iterrows():
    key = (cached["name_key"], str(cached.get("country", "")))
    if pd.notna(cached.get("latitude")) and pd.notna(cached.get("longitude")):
        cache_lookup[key] = (
            cached["latitude"], cached["longitude"],
            cached.get("geocode_query", "legacy cache"),
        )

new_cache_rows = []
latitudes, longitudes, geocode_sources = [], [], []
country_codes = {"CH": "ch", "IT": "it", "AT": "at", "FR": "fr"}

for _, row in df.iterrows():
    name_key = normalise_name(row["resort_name"])
    key = (name_key, str(row["country"]))
    force_refresh = name_key in regeocode_keys

    if key in cache_lookup and not force_refresh:
        lat, lon, _source = cache_lookup[key]
        latitudes.append(float(lat))
        longitudes.append(float(lon))
        geocode_sources.append("legacy cache (not independently revalidated)")
        continue

    query = REGEOCODE_QUERY_OVERRIDES.get(
        name_key, f"{row['resort_name']}, {row['country']}, Alps"
    )
    try:
        response = requests.get(
            "https://nominatim.openstreetmap.org/search",
            params={
                "q": query, "format": "jsonv2", "limit": 5,
                "addressdetails": 1, "countrycodes": country_codes[str(row["country"])],
            },
            headers=HEADERS,
            timeout=25,
        )
        response.raise_for_status()
        results = response.json()
        if results:
            country_matches = [
                item for item in results
                if item.get("address", {}).get("country_code", "").lower()
                == country_codes[str(row["country"])]
            ]
            selected = country_matches[0] if country_matches else results[0]
            lat, lon = float(selected["lat"]), float(selected["lon"])
            status = "matched"
            display_name = selected.get("display_name")
            osm_type = selected.get("osm_type")
            osm_id = selected.get("osm_id")
            osm_class = selected.get("class")
            osm_feature_type = selected.get("type")
        else:
            lat = lon = np.nan
            status = "not_found"
            display_name = osm_type = osm_id = osm_class = osm_feature_type = None
    except (requests.RequestException, ValueError, KeyError) as exc:
        lat = lon = np.nan
        status = f"request_error: {type(exc).__name__}"
        display_name = osm_type = osm_id = osm_class = osm_feature_type = None

    latitudes.append(lat)
    longitudes.append(lon)
    geocode_sources.append("Nominatim (refreshed)" if force_refresh else "Nominatim")
    new_cache_rows.append({
        "resort_name": row["resort_name"], "country": row["country"],
        "latitude": lat, "longitude": lon, "geocode_query": query,
        "geocode_status": status, "geocode_display_name": display_name,
        "geocode_osm_type": osm_type, "geocode_osm_id": osm_id,
        "geocode_class": osm_class, "geocode_type": osm_feature_type,
    })
    time.sleep(1.1)

df["latitude"] = latitudes
df["longitude"] = longitudes
df["geocode_source"] = geocode_sources

updated_cache = pd.concat(
    [geo_cache.drop(columns=["name_key"], errors="ignore"), pd.DataFrame(new_cache_rows)],
    ignore_index=True,
)
updated_cache = updated_cache.drop_duplicates(subset=["resort_name", "country"], keep="last")
updated_cache.to_csv(GEOCODE_CACHE_CSV, index=False)

print("Geocoded:", int(df[["latitude", "longitude"]].notna().all(axis=1).sum()), "/", len(df))
print("Coordinates sourced from legacy cache (not independently revalidated):",
      int(df["geocode_source"].str.startswith("legacy cache").sum()))
print("Refreshed with Nominatim:",
      int(df["geocode_source"].str.contains("refreshed", na=False).sum()))
print("Unresolved:")
display(df.loc[df["latitude"].isna() | df["longitude"].isna(), ["resort_name", "country"]])


## 2. Distances to major cities (Haversine)

Distances are straight-line great-circle distances, not driving distances. They are calculated only when both resort coordinates are available.

In [ ]:
for city, (city_lat, city_lon) in CITIES.items():
    df[f"dist_{city}_km"] = [
        round(haversine_km(lat, lon, city_lat, city_lon), 1) if pd.notna(lat) and pd.notna(lon) else np.nan
        for lat, lon in zip(df["latitude"], df["longitude"])
    ]

distance_columns = [f"dist_{city}_km" for city in CITIES]
df["dist_nearest_city_km"] = df[distance_columns].min(axis=1)
print("Resorts with a valid nearest-city distance:", int(df["dist_nearest_city_km"].notna().sum()))
display(df[["resort_name", "country", "dist_nearest_city_km"]].head(10))

## 3. Actual 4- and 5-star hotel counts from OpenStreetMap Overpass

The query includes hotel nodes, ways and relations. A 10 km Haversine buffer is applied to resort and hotel coordinates. The hotel cache records OSM element IDs and tags. If every Overpass endpoint fails, the notebook stops: it does not replace missing results with a slope/price proxy.

In [ ]:
OVERPASS_ENDPOINTS = [
    "https://overpass-api.de/api/interpreter",
    "https://overpass.kumi.systems/api/interpreter",
]

# Derive the query bounding box from the actual geocoded resort sample.
valid_coords = df.dropna(subset=["latitude", "longitude"])
if valid_coords.empty:
    raise RuntimeError("No resort coordinates available; cannot query nearby hotels.")
south = max(-90.0, float(valid_coords["latitude"].min()) - 0.20)
west = max(-180.0, float(valid_coords["longitude"].min()) - 0.20)
north = min(90.0, float(valid_coords["latitude"].max()) + 0.20)
east = min(180.0, float(valid_coords["longitude"].max()) + 0.20)
bbox = f"{south:.5f},{west:.5f},{north:.5f},{east:.5f}"
print("Overpass bounding box (south,west,north,east):", bbox)

OVERPASS_QUERY = f"""
[out:json][timeout:240];
(
  node["tourism"="hotel"]["stars"~"^(4|5)(\.0)?$"]({bbox});
  way["tourism"="hotel"]["stars"~"^(4|5)(\.0)?$"]({bbox});
  relation["tourism"="hotel"]["stars"~"^(4|5)(\.0)?$"]({bbox});
);
out center tags;
"""

hotel_elements = None
last_error = None
for endpoint in OVERPASS_ENDPOINTS:
    try:
        response = requests.post(
            endpoint, data={"data": OVERPASS_QUERY}, headers=HEADERS, timeout=270
        )
        response.raise_for_status()
        hotel_elements = response.json().get("elements", [])
        if hotel_elements:
            print(f"Retrieved {len(hotel_elements)} OSM hotel elements from {endpoint}")
            break
    except (requests.RequestException, ValueError) as exc:
        last_error = exc
        print(f"[WARN] Overpass endpoint failed: {endpoint} — {exc}")

if hotel_elements is None:
    raise RuntimeError(
        f"Overpass query failed on all endpoints. No proxy values were generated. Last error: {last_error}"
    )

hotel_rows = []
for element in hotel_elements:
    tags = element.get("tags", {})
    stars = str(tags.get("stars", "")).strip()
    if not re.fullmatch(r"[45](?:\.0)?", stars):
        continue
    lat = element.get("lat", element.get("center", {}).get("lat"))
    lon = element.get("lon", element.get("center", {}).get("lon"))
    if lat is None or lon is None:
        continue
    hotel_rows.append({
        "osm_type": element.get("type"),
        "osm_id": element.get("id"),
        "hotel_name": tags.get("name"),
        "stars": stars,
        "latitude": float(lat),
        "longitude": float(lon),
    })

hotels = pd.DataFrame(hotel_rows).drop_duplicates(subset=["osm_type", "osm_id"])
if hotels.empty:
    raise RuntimeError("Overpass responded, but no mappable 4- or 5-star hotels were returned.")
hotels["retrieved_at_utc"] = datetime.now(timezone.utc).isoformat()
hotels.to_csv(HOTEL_CACHE_CSV, index=False)
print("Unique mapped 4/5-star hotels:", len(hotels))

hotel_counts = []
for _, resort in df.iterrows():
    if pd.isna(resort["latitude"]) or pd.isna(resort["longitude"]):
        hotel_counts.append(np.nan)
        continue
    distances = haversine_km(
        resort["latitude"], resort["longitude"],
        hotels["latitude"].to_numpy(), hotels["longitude"].to_numpy()
    )
    hotel_counts.append(int(np.sum(distances <= 10.0)))

df["hotels_4_5_star_10km"] = hotel_counts
df["hotel_data_source"] = pd.Series(pd.NA, index=df.index, dtype="string")
df.loc[df["hotels_4_5_star_10km"].notna(), "hotel_data_source"] = "OpenStreetMap Overpass"
print("Resorts with hotel counts:",
      int(df["hotels_4_5_star_10km"].notna().sum()), "/", len(df))


## 4. EDA — prices and resort size

The analysis uses observed prices only. Swiss prices without an official approximate EUR value are left missing. The plots show the distribution of daily ticket prices by country, slope kilometres by country, and the relationship between resort size and price.

In [ ]:
# Descriptive statistics
eda_summary = df.groupby("country").agg(
    resorts=("resort_name", "count"),
    mean_price_eur=("price_day_eur", "mean"),
    median_price_eur=("price_day_eur", "median"),
    mean_slope_km=("slope_km", "mean"),
    median_slope_km=("slope_km", "median"),
    mean_lifts=("lifts_count", "mean"),
)
print(eda_summary.round(2))

fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column="price_day_eur", by="country", ax=ax)
ax.set_title("Daily ski-pass price by country (EUR)")
ax.set_xlabel("Country")
ax.set_ylabel("Daily price (EUR)")
fig.suptitle("")
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
df.boxplot(column="slope_km", by="country", ax=ax)
ax.set_title("Ski slope length by country")
ax.set_xlabel("Country")
ax.set_ylabel("Total slope length (km)")
fig.suptitle("")
plt.tight_layout()
plt.show()

plot_data = df.dropna(subset=["slope_km", "price_day_eur"])
fig, ax = plt.subplots(figsize=(9, 5))
for country, group in plot_data.groupby("country"):
    ax.scatter(group["slope_km"], group["price_day_eur"], label=country, alpha=0.75)
ax.set_title("Resort size and daily ticket price")
ax.set_xlabel("Total slope length (km)")
ax.set_ylabel("Daily price (EUR)")
ax.legend(title="Country")
plt.tight_layout()
plt.show()

## 5. Export and final checks

The feature dataset retains all 280 resort rows. Unresolved geocodes and unavailable hotel counts remain `NaN`; this is preferable to false precision. The summary below reports coverage explicitly.

In [ ]:
df.to_csv(FEATURES_CSV, index=False)

coverage = pd.DataFrame({
    "metric": ["coordinates", "nearest-city distance", "hotel counts", "EUR ticket price", "slope length"],
    "available_rows": [
        int(df[["latitude", "longitude"]].notna().all(axis=1).sum()),
        int(df["dist_nearest_city_km"].notna().sum()),
        int(df["hotels_4_5_star_10km"].notna().sum()),
        int(df["price_day_eur"].notna().sum()),
        int(df["slope_km"].notna().sum()),
    ],
})
coverage["total_rows"] = len(df)
coverage["coverage_pct"] = (coverage["available_rows"] / len(df) * 100).round(1)
print("Saved:", FEATURES_CSV)
print("Rows:", len(df))
print("Country distribution:")
print(df.groupby("country").size())
print("\nFeature coverage:")
display(coverage)
display(df.head(10))